In [ ]:
#更新資料
%reload_ext autoreload
%autoreload 2
import Stock
import pandas as pd

stock=[]

if 0:
    #從html,pdf中抓取資料
    Stock.import_data(stock)
    stock=pd.DataFrame(data=stock).replace(['NaN', 'nan', 'None', '', ' '], 0)
    stock.to_csv("./csv/Stock_History.csv", encoding='utf-8')

stock = pd.read_csv("./csv/Stock_History.csv",index_col=0,encoding='utf-8')
dividends_file = pd.read_csv("./csv/Dividends_History.csv",index_col=0,encoding='utf-8')
stock_not_in_file = pd.read_csv("./csv/Stock_not_in_file.csv",index_col=0,encoding='utf-8')
combined_df = pd.concat([stock, dividends_file, stock_not_in_file], axis=0)
combined_df.index = pd.to_datetime(combined_df['date'])
stock = combined_df.sort_index().to_dict('records')[1:]


sold,buy,fee,discount,pvalue,box1 = Stock.PortfolioCalculator.calculate_metrics(stock)
box,pvalue,inventory,cost,df_sold,dividends = Stock.refresh_history(box1,stock)
profit_breakdown = Stock.PortfolioCalculator.calculate_profit_breakdown(stock, box)
holding_details = Stock.PortfolioCalculator.calculate_holding_details(stock, inventory, box)
holding_details

In [ ]:
#畫圖
%reload_ext autoreload
%autoreload 2
import Stock

stock=pd.DataFrame(data=stock)
df_xirr=pd.DataFrame()
stock['net_cash'] = stock['notional'] - (stock['fee'] - stock['discount'])
stock.loc[stock['type'] == '現賣', 'net_cash'] -= stock['tax']
stock.loc[stock['type'] == '現買', 'net_cash'] *= -1
df_xirr['net_cash'] = stock['net_cash']
df_xirr['date'] = stock['date']
df_xirr.to_csv('test.csv')
risk_free_rate = 0.018

fw = Stock.show_2_chart(box,inventory,cost,df_sold,dividends,df_xirr,profit_breakdown)
fw

In [ ]:
# 依最新實際持倉計算 Sharpe ratio
import numpy as np
import pandas as pd

# 1. 從最新每日市值取得非零持倉及正規化權重
daily_worth = pd.read_csv('./csv/Daily_Worth.csv', index_col=0, parse_dates=True)
latest_values = daily_worth.drop(columns=['grand_total', '^VIX'], errors='ignore').iloc[-1]
latest_values = latest_values[latest_values > 0].astype(float)
weights = latest_values / latest_values.sum()

# 2. 使用與投資組合估值一致的本地價格資料
all_prices = pd.read_csv('./csv/All_Price.csv', index_col=0, parse_dates=True)
missing_assets = weights.index.difference(all_prices.columns)
if not missing_assets.empty:
    raise ValueError(f'All_Price.csv 缺少持倉價格欄位: {missing_assets.tolist()}')
price_data = all_prices[weights.index].ffill().dropna()
returns = price_data.pct_change().dropna()
if len(returns) < 2:
    raise ValueError('共同價格資料不足，無法計算 Sharpe ratio')
returns.to_csv('test.csv')

# 3. 依實際持倉權重計算年化報酬、波動與 Sharpe ratio
risk_free_rate = 0.016  # 假設年化無風險利率 1.6%
port_return = float(returns.mean().dot(weights) * 252)
port_volatility = float(np.sqrt(weights.T.dot(returns.cov() * 252).dot(weights)))
sharpe_ratio = (port_return - risk_free_rate) / port_volatility

print('持倉權重:')
print((weights * 100).round(2).astype(str) + '%')
print(f'計算期間: {returns.index[0].date()} 至 {returns.index[-1].date()}')
print(f'組合年化報酬率: {port_return:.2%}')
print(f'組合年化波動率: {port_volatility:.2%}')
print(f'組合年化 Sharpe ratio: {sharpe_ratio:.2f}')